In [ ]:
# required packages

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import scipy as sc
import pandas as pd
import networkx as nx

from scipy import interpolate
from scipy.io import savemat
from scipy.stats import ttest_ind
from sklearn.metrics import mean_squared_error
from scipy import sparse
from pynwb import NWBHDF5IO


%matplotlib inline

In [ ]:
# load in desired .nwb file

nwb_filepath = ""

io = NWBHDF5IO(nwb_filepath, mode="r")
nwb = io.read()

In [ ]:
# RRR class used in this project, originally implemented by Chris Rayner

"""
Reduced rank regression class.
Requires scipy to be installed.

Implemented by Chris Rayner (2015)
dchrisrayner AT gmail DOT com

Optimal linear 'bottlenecking' or 'multitask learning'.
"""

class RRR(object):
    """
    Reduced Rank Regressor (linear 'bottlenecking' or 'multitask learning')
    - X is an n-by-d matrix of features.
    - Y is an n-by-D matrix of targets.
    - rrank is a rank constraint.
    - reg is a regularization parameter (optional).
    """
    def __init__(self, X, Y, rank, reg=None):
        if np.size(np.shape(X)) == 1:
            X = np.reshape(X, (-1, 1))
        if np.size(np.shape(Y)) == 1:
            Y = np.reshape(Y, (-1, 1))
        if reg is None:
            reg = 0
        self.rank = rank

        CXX = np.dot(X.T, X) + reg * sparse.eye(np.size(X, 1))
        CXY = np.dot(X.T, Y)
        _U, _S, V = np.linalg.svd(np.dot(CXY.T, np.dot(np.linalg.pinv(CXX), CXY)))
        self.W = V[0:rank, :].T
        self.A = np.dot(np.linalg.pinv(CXX), np.dot(CXY, self.W)).T
        self.V = V 


    def __str__(self):
        return 'Reduced Rank Regressor (rank = {})'.format(self.rank)

    def predict(self, X):
        """Predict Y from X."""
        if np.size(np.shape(X)) == 1:
            X = np.reshape(X, (-1, 1))
        return np.dot(X, np.dot(self.A.T, self.W.T))

    # this function retrieves the V matrix
    def get_V(self):
        return self.V

    def get_R2(self):
        return self.R2

    def get_R2_ridge(self):
        return self.R2_ridge

In [ ]:
# function used to quantify error in the model

def sqerr(matrix1, matrix2):
    # returning r^2 as the error metric
    r = np.corrcoef(matrix1.flatten(), matrix2.flatten())[0][1]
    return r**2

In [ ]:
# compute a folded cross validation and train the model using RRR

def folded_Reduced_Rank(matrix1, matrix2, nfold=10, ptype='regular'):
    if ptype == 'causal':
        nbins = 9
    else:
        nbins = 10
    # regularization, need to adjust parameters of np.arange() depending on regularization range
    lamvals  = np.arange(0, 0.2, 0.05)
    RANK = 25
    SPLIT  = len(matrix1[0])//nbins//nfold*nbins
    matrix1, matrix2 = matrix1[:, :nfold*SPLIT], matrix2[:, :nfold*SPLIT]
    XX, YY = matrix1.T, matrix2.T
    V = []
    prediction_err, predictive_dim = [], []
    for fold in range(nfold):
        initial_index, final_index = fold*SPLIT, (fold+1)*SPLIT
        testXX,  testYY  = XX[initial_index:final_index], YY[initial_index:final_index]
        trainXX, trainYY = np.delete(XX, np.arange(initial_index, final_index), axis=0), np.delete(YY, np.arange(initial_index, final_index), axis=0)
        training_error, testing_error = [], []
        # uncomment below to run the lambda selection; current version uses no regularization
        '''
        for LAMBDA in lamvals:
            regressor = RRR(trainXX, trainYY, RANK, LAMBDA)
            #R2.append(regressor.get_R2())
            #R2_ridge.append(regressor.get_R2_ridge())
            #print(R2)
            #print(R2_ridge)
            training_error.append(sqerr(regressor.predict(trainXX), trainYY))
            testing_error.append(sqerr(regressor.predict(testXX), testYY))
        #bestLAM  = lamvals[np.argmax(testing_error)]'''
        bestLAM = 0
        
        # train the model with the selected lambda and calculate error
        regressor = RRR(trainXX, trainYY, RANK, bestLAM)
        R2.append(regressor.get_R2())
        R2_ridge.append(regressor.get_R2_ridge())
        V.append(regressor.get_V())
        CXX = np.dot(trainXX.T, trainXX) + bestLAM * sparse.eye(np.size(trainXX, 1))
        CXY = np.dot(trainXX.T, trainYY)
        PE  = []

        # iterate for every rank
        for rank in range(1, 26):
            W = V[fold][:rank, :].T  
            A = np.dot(np.linalg.pinv(CXX), np.dot(CXY, W)).T  
            pred = np.dot(trainXX, np.dot(A.T, W.T)) 
            PE.append(sqerr(pred, trainYY))

       
        prediction_err.append(PE)
        predictive_dim.append(A)

    return V, prediction_err, predictive_dim

In [ ]:
def performance_curve(area1, area2, N, title1, title2):
    
    # calculate the performance curve (R^2 vs. number of ranks) for two areas

    # load in the dff data for area 1
    dff_1 = nwb.processing[area1]["dff"]
    dff_trace_1 = np.array(dff_1.roi_response_series["dff_timeseries"].data)
    dff_timestamps_1 = np.array(dff_1.roi_response_series["dff_timeseries"].timestamps)

    # load in the dff data for area 2
    dff_2 = nwb.processing[area2]["dff"]
    dff_trace_2 = np.array(dff_2.roi_response_series["dff_timeseries"].data)
    dff_timestamps_2 = np.array(dff_2.roi_response_series["dff_timeseries"].timestamps)

    # initialize error lists
    all_avg_errors = []
    all_std_errors = []

    # loop over all ranks
    for _ in range(N):
        # Randomly select 25 neurons from the first area
        selected_indices_1 = np.random.choice(dff_trace_1.shape[1], 25, replace=False)

        # conditional statement, if the areas are the same select distinct (but still random) neuron sets
        if area1 == area2:
            remaining_indices = np.setdiff1d(np.arange(dff_trace_1.shape[1]), selected_indices_1)
            selected_indices_2 = np.random.choice(remaining_indices, 25, replace=False)
        # if areas are different, can pick any random 25 cells from area 2
        else:
            selected_indices_2 = np.random.choice(dff_trace_2.shape[1], 25, replace=False)

        # extract the selected neurons and assign them to matrices before performing RRR
        matrix1 = dff_trace_1[:, selected_indices_1].T
        matrix2 = dff_trace_2[:, selected_indices_2].T

        # run RRR
        V, prediction_err, predictive_dim = folded_Reduced_Rank(matrix1, matrix2, nfold=10, ptype='regular')

        # compute average error across folds
        avg_error = np.mean(np.array(prediction_err), axis=0)
        std_error = np.std(np.array(prediction_err), axis=0)

        # store the fold-averaged error for this iteration
        all_avg_errors.append(avg_error)
        all_std_errors.append(std_error)

    # convert to numpy arrays
    all_avg_errors = np.array(all_avg_errors)
    all_std_errors = np.array(all_std_errors)

    # compute standard error (to build confidence interval)
    sem_errors = all_std_errors / np.sqrt(N)

    # calculate 95% confidence interval (with standard z-score of 1.96 for this confidence interval)
    ci_upper = all_avg_errors + 1.96 * sem_errors
    ci_lower = all_avg_errors - 1.96 * sem_errors

    # final error is average over all trials (i.e., average over each N)
    final_avg_error = np.mean(all_avg_errors, axis=0)
    final_ci_upper = np.mean(ci_upper, axis=0)
    final_ci_lower = np.mean(ci_lower, axis=0)

    # only plotting the first 10 ranks (saturation value is reached before 10 so no need to plot after)
    ranks = np.arange(1, 11)

    # because we're only plotting the first 10 ranks, only need the first 10 errors and confidence intervals 
    final_avg_error = final_avg_error[0:10]
    final_ci_upper = final_ci_upper[0:10]
    final_ci_lower = final_ci_lower[0:10]

    plt.fill_between(ranks, final_ci_lower, final_ci_upper, color='b', alpha=0.1, label='95% CI')
    plt.plot(ranks, final_avg_error, '-o')
    plt.xlabel("Rank")
    plt.ylabel(r'$R^2$')
    plt.title(f"Performance Curve: {title1} to {title2}, N = {N}")
    plt.xticks(ranks)
    plt.legend()
    plt.show()

    # calculate the saturation value of the curve, and the rank at which 90% of the saturation value is achieved
    saturation_value = np.max(final_avg_error)
    target_value = 0.9 * saturation_value
    closest_index = np.argmin(np.abs(final_avg_error - target_value))
    low_rank = ranks[closest_index]

    return saturation_value, low_rank